# Qwen3-8B: Tamil rerun (ta, ta_en) at batch 1

The all-language run hit CUDA OOM on Tamil at batch 8. Starts from `rajtaware10/safl-dpi-all-qwen` outputs; drops the partial Tamil meta and its judge rows.

In [ ]:
# The GitHub repo is private, so the code comes in as a private Kaggle dataset
# (dataset_sources in kernel-metadata.json, or Add Input in the UI).
import glob, os, shutil, zipfile
!nvidia-smi -L
W = "/kaggle/working/safl-hackathon"
hits = glob.glob("/kaggle/input/**/collect.py", recursive=True)
if hits:
    shutil.copytree(os.path.dirname(hits[0]), W, dirs_exist_ok=True)
else:
    zips = glob.glob("/kaggle/input/**/code.zip", recursive=True)
    assert zips, "code dataset not attached: nothing under /kaggle/input"
    zipfile.ZipFile(zips[0]).extractall(W)
%cd {W}
!ls -R | head -40

In [ ]:
# Gemma-4 (dhee-pravega) needs a recent transformers; 5.7.0 is the version checked locally.
!pip install -q "transformers==5.7.0" accelerate
import transformers, torch
print(transformers.__version__, torch.__version__, torch.cuda.device_count(), "GPUs")

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set from Kaggle secrets")
except Exception as e:
    print("No HF_TOKEN secret; public downloads only:", type(e).__name__)

In [ ]:
!mkdir -p /kaggle/working/logs && df -h / /kaggle/working | tail -2

In [ ]:
import glob, zipfile, json, os
z = glob.glob('/kaggle/input/**/all_outputs_qwen.zip', recursive=True)
assert z, 'attach kernel output rajtaware10/safl-dpi-all-qwen'
zipfile.ZipFile(z[0]).extractall('.')
d = 'data/qwen3-8b'
for f in ('ta_meta.jsonl', 'ta_acts.npy', 'ta_en_meta.jsonl', 'ta_en_acts.npy'):
    if os.path.exists(f'{d}/{f}'): os.remove(f'{d}/{f}')
rows = [l for l in open(f'{d}/judge_scores.jsonl', encoding='utf-8') if json.loads(l)['lang'] not in ('ta', 'ta_en')]
open(f'{d}/judge_scores.jsonl', 'w', encoding='utf-8').writelines(rows)
print(len(rows), 'judge rows kept'); print(sorted(os.listdir(d)))

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python collect.py --model qwen3-8b --langs ta ta_en --batch-size 1 2>&1 | tee /kaggle/working/logs/collect_qwen_ta.log

In [ ]:
!rm -rf ~/.cache/huggingface/hub/models--Qwen--Qwen3-8B; df -h / | tail -1

In [ ]:
!pip install -q bitsandbytes
!python judge.py --agent qwen3-8b --langs ta ta_en --load-4bit 2>&1 | tee /kaggle/working/logs/judge_qwen_ta.log

In [ ]:
!zip -q -r /kaggle/working/all_outputs_qwen_ta.zip data/qwen3-8b && ls -la /kaggle/working/all_outputs_qwen_ta.zip